# Robustness / Sensitivity Analysis 
**ABa-KiTo | All three scenarios**

This notebook builds the stability table for the paper, scenario by scenario.
It mirrors the data-loading pattern of `2025-11-20-ABa-KiTo-N4intervals-graphAnalyisis.ipynb`.

**What we report per run (no heuristic basin counts):**
- Total clusters
- Noise %
- `chi_min` — average χ of the lowest cluster (brown macrostate anchor)
- `chi_max` — average χ of the highest cluster (green macrostate anchor)

**Structure:**
1. Setup & imports
2. Scenario 1 — Voter dynamics, fully connected (N=3 and N=4)
3. Scenario 2 — Voter dynamics, small-world (N=3)
4. Scenario 3 — Majority rule (N=3)
5. Compile full DataFrame
6. Summary table
7. LaTeX table for paper

---
## 1. Setup & imports

In [1]:
import numpy as np
import pandas as pd
import pickle
import os
import sys

# ── Point to the project root (two levels up from any scenario notebook) ──
# Adjust depth if you place this notebook somewhere else
project_root = os.path.abspath(os.path.join(os.getcwd(), '..') )
if project_root not in sys.path:
    sys.path.append(project_root)

print(f'Project root: {project_root}')
print(f'Contents: {os.listdir(project_root)}')

Project root: c:\Users\Flo\Documents\Sarai\ABa-KiTo-V02
Contents: ['.DS_Store', '.git', '.vscode', 'ABa-KiTo', 'desktop.ini', 'Manifest.toml', 'Project.toml', 'README.md', 'src']


In [2]:
# ── Helper: compute per-cluster chi statistics from FNs + chi array ──
# This mirrors exactly what the stability notebook does:
#   chi_means[node_id] = np.mean(data.chi0[FNs.nodes == node_id])

def cluster_stats(FNs, chi0):
    """
    Given a loaded FNs object and the flat chi0 array,
    returns a dict with the robustness metrics for one run.
    """
    chi_means = []
    for node_id in range(FNs.Nnodes):
        mask = FNs.nodes == node_id
        chi_means.append(float(np.mean(chi0[mask])))
    chi_means = np.array(chi_means)

    total     = np.sum(FNs.nodes >= -1)   # all points including noise
    noise_n   = np.sum(FNs.nodes == -1)
    noise_pct = 100.0 * noise_n / total

    return {
        'n_clusters': int(FNs.Nnodes),
        'noise_pct':  round(float(noise_pct), 1),
        'chi_min':    round(float(chi_means.min()), 3),
        'chi_max':    round(float(chi_means.max()), 3),
        'chi_per_cluster': chi_means.tolist(),   # useful for inspection
    }


def load_fns(pkl_path):
    """Load a saved FNs pickle."""
    with open(pkl_path, 'rb') as f:
        return pickle.load(f)


print('Helpers defined.')

Helpers defined.


---
## 2. Scenario 1 — Voter dynamics, fully connected

**Paper configuration:** N=4, seed=123  
**Stability sweep:** N ∈ {3, 4}, seeds ∈ {42, 123, 456}  
**Pickle location:** `s01-voter_dynamics_FC/003-clustering/stability_results/`

In [3]:
# ── S1: Load chi values ──────────────────────────────────────────────────────
# Same loading code as in the graph analysis notebook

s1_root   = os.path.join(project_root, 'ABa-KiTo', 's01-voter_dynamics_FC')
data_dir  = os.path.join(s1_root, 'data', '')

# Load states (needed to build scaler — required to reconstruct data object if needed)
states_data = np.load(data_dir + 'simulations/2025-10-20-data_VD_FC_3InteractingAgents_rand_init_capital.npz')

# Load chi values
chi_npz = np.load(data_dir + 'chi_vals/chi_values_rand_init_capitals.npz')

# Inspect what keys are in the file — run this to confirm
print('Keys in chi npz:', list(chi_npz.keys()) if hasattr(chi_npz, 'keys') else 'plain array')
print('Type:', type(chi_npz))

Keys in chi npz: plain array
Type: <class 'numpy.ndarray'>


In [4]:
# ── S1: Extract the flat chi array ───────────────────────────────────────────
# Replace 'chi' below with whatever key printed above (e.g. 'chi0', 'arr_0', etc.)
# If it printed 'plain array', use: chi0_s1 = chi_npz.ravel()

chi0_s1 = chi_npz.ravel()      # <── adjust key if needed

print(f'chi0_s1 shape : {chi0_s1.shape}')
print(f'chi0_s1 range : [{chi0_s1.min():.3f}, {chi0_s1.max():.3f}]')

chi0_s1 shape : (200000,)
chi0_s1 range : [-0.013, 1.032]


In [5]:
# ── S1: Loop over all (N, seed) configurations ───────────────────────────────

s1_pkl_dir = os.path.join(s1_root, '003-clustering', 'stability_results')

N_intervals_s1 = [3, 4]      # 3 = alternative, 4 = paper choice
seeds          = [42, 123, 456]
paper_config   = (4, 123)    # (N, seed) used in the paper

records_s1 = []

for N in N_intervals_s1:
    for seed in seeds:
        pkl_path = os.path.join(s1_pkl_dir, f'Nint{N}_seed{seed}_FNs.pkl')
        
        if not os.path.exists(pkl_path):
            print(f'  MISSING: {pkl_path}')
            continue
        
        FNs    = load_fns(pkl_path)
        stats  = cluster_stats(FNs, chi0_s1)
        chosen = (N, seed) == paper_config
        
        records_s1.append({
            'scenario':    'S1',
            'N_intervals': N,
            'seed':        seed,
            'chosen':      chosen,
            **stats
        })
        
        marker = '  [PAPER]' if chosen else ''
        print(f'  N={N} seed={seed}{marker} | '
              f'clusters={stats["n_clusters"]} '
              f'noise={stats["noise_pct"]:.1f}% '
              f'chi=[{stats["chi_min"]:.3f}, {stats["chi_max"]:.3f}]')

df_s1 = pd.DataFrame(records_s1)
print(f'\nS1 records collected: {len(df_s1)}')
df_s1.drop(columns=['chi_per_cluster']).round(3)

  N=3 seed=42 | clusters=6 noise=6.6% chi=[0.161, 0.694]
  N=3 seed=123 | clusters=4 noise=35.3% chi=[0.161, 0.692]
  N=3 seed=456 | clusters=6 noise=6.6% chi=[0.161, 0.694]
  N=4 seed=42 | clusters=11 noise=7.4% chi=[0.137, 0.713]
  N=4 seed=123  [PAPER] | clusters=10 noise=7.4% chi=[0.136, 0.713]
  N=4 seed=456 | clusters=10 noise=7.4% chi=[0.136, 0.713]

S1 records collected: 6


,scenario,N_intervals,seed,chosen,n_clusters,noise_pct,chi_min,chi_max
0,S1,3,42,False,6,6.6,0.161,0.694
1,S1,3,123,False,4,35.3,0.161,0.692
2,S1,3,456,False,6,6.6,0.161,0.694
3,S1,4,42,False,11,7.4,0.137,0.713
4,S1,4,123,True,10,7.4,0.136,0.713
5,S1,4,456,False,10,7.4,0.136,0.713


In [6]:
# ── S1: Inspect chi values per cluster for the paper configuration ───────────
# Useful to visually confirm brown and green endpoints

paper_row = df_s1[(df_s1.N_intervals == 4) & (df_s1.seed == 123)].iloc[0]
print('Chi value per cluster (N=4, seed=123):')
for i, v in enumerate(paper_row['chi_per_cluster']):
    print(f'  Cluster {i:2d}: chi_mean = {v:.3f}')

Chi value per cluster (N=4, seed=123):
  Cluster  0: chi_mean = 0.136
  Cluster  1: chi_mean = 0.144
  Cluster  2: chi_mean = 0.246
  Cluster  3: chi_mean = 0.315
  Cluster  4: chi_mean = 0.282
  Cluster  5: chi_mean = 0.397
  Cluster  6: chi_mean = 0.421
  Cluster  7: chi_mean = 0.446
  Cluster  8: chi_mean = 0.713
  Cluster  9: chi_mean = 0.656


---
## 3. Scenario 2 — Voter dynamics, small-world

**Paper configuration:** N=3, seed=123  
**Pickle location:** `s02-voter_dynamics_SW/data/clustering_results/`

In [7]:
# ── S2: Load chi values ──────────────────────────────────────────────────────

s2_root  = os.path.join(project_root, 'ABa-KiTo', 's02-voter_dynamics_SW')
chi_dir2 = os.path.join(s2_root, 'data', 'chi_vals')

# List what is in the chi_vals folder so we can confirm the filename
print('Files in chi_vals/:')
for f in os.listdir(chi_dir2):
    print(f'  {f}')

Files in chi_vals/:
  chi_values_VD_SW.npz


In [8]:
# ── S2: Load chi — update filename from the list above ───────────────────────

chi_path_s2 = os.path.join(chi_dir2, 'chi_values_VD_SW.npz')   # <── update if needed

chi_raw_s2 = np.load(chi_path_s2, allow_pickle=True)
print('Type:', type(chi_raw_s2))

# If NpzFile:
if hasattr(chi_raw_s2, 'keys'):
    print('Keys:', list(chi_raw_s2.keys()))
    chi0_s2 = chi_raw_s2[list(chi_raw_s2.keys())[0]].ravel()  # take first key
else:
    # Plain array
    chi0_s2 = chi_raw_s2.ravel()

print(f'chi0_s2 shape : {chi0_s2.shape}')
print(f'chi0_s2 range : [{chi0_s2.min():.3f}, {chi0_s2.max():.3f}]')

Type: <class 'numpy.ndarray'>
chi0_s2 shape : (200000,)
chi0_s2 range : [0.002, 1.020]


In [9]:
# ── S2: Loop over all (N, seed) configurations ───────────────────────────────

s2_pkl_dir = os.path.join(s2_root, 'data', 'clustering_results')

print('Pickle files found in clustering_results/:')
for f in os.listdir(s2_pkl_dir):
    print(f'  {f}')

Pickle files found in clustering_results/:
  Nint2_seed123_FCs.pkl
  Nint2_seed123_FNs.pkl
  Nint2_seed42_FCs.pkl
  Nint2_seed42_FNs.pkl
  Nint2_seed456_FCs.pkl
  Nint2_seed456_FNs.pkl
  Nint3_seed123_FCs.pkl
  Nint3_seed123_FNs.pkl
  Nint3_seed42_FCs.pkl
  Nint3_seed42_FNs.pkl
  Nint3_seed456_FCs.pkl
  Nint3_seed456_FNs.pkl
  Nint4_seed123_FCs.pkl
  Nint4_seed123_FNs.pkl
  Nint4_seed42_FCs.pkl
  Nint4_seed42_FNs.pkl
  Nint4_seed456_FCs.pkl
  Nint4_seed456_FNs.pkl
  summary.csv


In [10]:
N_intervals_s2 = [3]         # paper uses 3; add 4 here once you run that sweep
seeds          = [42, 123, 456]
paper_config_s2 = (3, 123)

records_s2 = []

for N in N_intervals_s2:
    for seed in seeds:
        pkl_path = os.path.join(s2_pkl_dir, f'Nint{N}_seed{seed}_FNs.pkl')
        
        if not os.path.exists(pkl_path):
            print(f'  MISSING: {os.path.basename(pkl_path)}')
            continue
        
        FNs    = load_fns(pkl_path)
        stats  = cluster_stats(FNs, chi0_s2)
        chosen = (N, seed) == paper_config_s2
        
        records_s2.append({
            'scenario':    'S2',
            'N_intervals': N,
            'seed':        seed,
            'chosen':      chosen,
            **stats
        })
        
        marker = '  [PAPER]' if chosen else ''
        print(f'  N={N} seed={seed}{marker} | '
              f'clusters={stats["n_clusters"]} '
              f'noise={stats["noise_pct"]:.1f}% '
              f'chi=[{stats["chi_min"]:.3f}, {stats["chi_max"]:.3f}]')

df_s2 = pd.DataFrame(records_s2)
print(f'\nS2 records collected: {len(df_s2)}')
df_s2.drop(columns=['chi_per_cluster'], errors='ignore').round(3)

  N=3 seed=42 | clusters=13 noise=7.5% chi=[0.202, 0.697]
  N=3 seed=123  [PAPER] | clusters=13 noise=7.5% chi=[0.201, 0.692]
  N=3 seed=456 | clusters=13 noise=7.5% chi=[0.201, 0.692]

S2 records collected: 3


,scenario,N_intervals,seed,chosen,n_clusters,noise_pct,chi_min,chi_max
0,S2,3,42,False,13,7.5,0.202,0.697
1,S2,3,123,True,13,7.5,0.201,0.692
2,S2,3,456,False,13,7.5,0.201,0.692


In [11]:
# ── S2: Loop over all (N, seed) configurations ───────────────────────────────

s2_pkl_dir = os.path.join(s2_root, 'data', 'clustering_results/')

N_intervals_s2 = [3, 4]      # 3 = alternative, 4 = paper choice
seeds          = [42, 123, 456]
paper_config   = (3, 123)    # (N, seed) used in the paper

records_s2 = []

for N in N_intervals_s2:
    for seed in seeds:
        pkl_path = os.path.join(s2_pkl_dir, f'Nint{N}_seed{seed}_FNs.pkl')
        
        if not os.path.exists(pkl_path):
            print(f'  MISSING: {pkl_path}')
            continue
        
        FNs    = load_fns(pkl_path)
        stats  = cluster_stats(FNs, chi0_s2)
        chosen = (N, seed) == paper_config
        
        records_s2.append({
            'scenario':    'S2',
            'N_intervals': N,
            'seed':        seed,
            'chosen':      chosen,
            **stats
        })
        
        marker = '  [PAPER]' if chosen else ''
        print(f'  N={N} seed={seed}{marker} | '
              f'clusters={stats["n_clusters"]} '
              f'noise={stats["noise_pct"]:.1f}% '
              f'chi=[{stats["chi_min"]:.3f}, {stats["chi_max"]:.3f}]')

df_s2 = pd.DataFrame(records_s2)
print(f'\nS1 records collected: {len(df_s2)}')
df_s2.drop(columns=['chi_per_cluster']).round(3)

  N=3 seed=42 | clusters=13 noise=7.5% chi=[0.202, 0.697]
  N=3 seed=123  [PAPER] | clusters=13 noise=7.5% chi=[0.201, 0.692]
  N=3 seed=456 | clusters=13 noise=7.5% chi=[0.201, 0.692]
  N=4 seed=42 | clusters=25 noise=7.0% chi=[0.181, 0.746]
  N=4 seed=123 | clusters=17 noise=7.1% chi=[0.184, 0.752]
  N=4 seed=456 | clusters=26 noise=7.0% chi=[0.181, 0.746]

S1 records collected: 6


,scenario,N_intervals,seed,chosen,n_clusters,noise_pct,chi_min,chi_max
0,S2,3,42,False,13,7.5,0.202,0.697
1,S2,3,123,True,13,7.5,0.201,0.692
2,S2,3,456,False,13,7.5,0.201,0.692
3,S2,4,42,False,25,7.0,0.181,0.746
4,S2,4,123,False,17,7.1,0.184,0.752
5,S2,4,456,False,26,7.0,0.181,0.746


---
## 4. Scenario 3 — Majority rule

**Paper configuration:** N=3, seed=123  
**Pickle location:** `s03-majority_rule/data/clustering_results/`

In [12]:
# ── S3: Load chi values ──────────────────────────────────────────────────────

s3_root  = os.path.join(project_root, 'ABa-KiTo', 's03-majority_rule')
chi_dir3 = os.path.join(s3_root, 'data', 'chi_vals')

# List files to confirm the chi filename
print('Files in chi_vals/:')
for f in os.listdir(chi_dir3):
    print(f'  {f}')

Files in chi_vals/:
  chi_values_MR.npz


In [13]:
# ── S3: Load chi — update filename from the list above ───────────────────────

chi_path_s3 = os.path.join(chi_dir3, 'chi_values_MR.npz')  

chi_raw_s3 = np.load(chi_path_s3, allow_pickle=True)
print('Type:', type(chi_raw_s3))

if hasattr(chi_raw_s3, 'keys'):
    print('Keys:', list(chi_raw_s3.keys()))
    chi0_s3 = chi_raw_s3[list(chi_raw_s3.keys())[0]].ravel()
else:
    chi0_s3 = chi_raw_s3.ravel()

print(f'chi0_s3 shape : {chi0_s3.shape}')
print(f'chi0_s3 range : [{chi0_s3.min():.3f}, {chi0_s3.max():.3f}]')

Type: <class 'numpy.ndarray'>
chi0_s3 shape : (200000,)
chi0_s3 range : [-0.003, 1.026]


In [14]:
# ── S3: Loop over all (N, seed) configurations ───────────────────────────────

s3_pkl_dir = os.path.join(s3_root, 'data', 'clustering_results')

print('Pickle files found in clustering_results/:')
for f in os.listdir(s3_pkl_dir):
    print(f'  {f}')

Pickle files found in clustering_results/:
  h_logistic
  Nint3_seed123_FCs.pkl
  Nint3_seed123_FNs.pkl
  Nint3_seed42_FCs.pkl
  Nint3_seed42_FNs.pkl
  Nint3_seed456_FCs.pkl
  Nint3_seed456_FNs.pkl
  Nint4_seed123_FCs.pkl
  Nint4_seed123_FNs.pkl
  Nint4_seed42_FCs.pkl
  Nint4_seed42_FNs.pkl
  Nint4_seed456_FCs.pkl
  Nint4_seed456_FNs.pkl
  summary.csv


In [15]:
N_intervals_s3  = [3]
seeds           = [42, 123, 456]
paper_config_s3 = (3, 123)

records_s3 = []

for N in N_intervals_s3:
    for seed in seeds:
        pkl_path = os.path.join(s3_pkl_dir, f'Nint{N}_seed{seed}_FNs.pkl')
        
        if not os.path.exists(pkl_path):
            print(f'  MISSING: {os.path.basename(pkl_path)}')
            continue
        
        FNs    = load_fns(pkl_path)
        stats  = cluster_stats(FNs, chi0_s3)
        chosen = (N, seed) == paper_config_s3
        
        records_s3.append({
            'scenario':    'S3',
            'N_intervals': N,
            'seed':        seed,
            'chosen':      chosen,
            **stats
        })
        
        marker = '  [PAPER]' if chosen else ''
        print(f'  N={N} seed={seed}{marker} | '
              f'clusters={stats["n_clusters"]} '
              f'noise={stats["noise_pct"]:.1f}% '
              f'chi=[{stats["chi_min"]:.3f}, {stats["chi_max"]:.3f}]')

df_s3 = pd.DataFrame(records_s3)
print(f'\nS3 records collected: {len(df_s3)}')
df_s3.drop(columns=['chi_per_cluster'], errors='ignore').round(3)

  N=3 seed=42 | clusters=7 noise=7.5% chi=[0.134, 0.861]
  N=3 seed=123  [PAPER] | clusters=7 noise=7.5% chi=[0.134, 0.861]
  N=3 seed=456 | clusters=7 noise=7.5% chi=[0.134, 0.861]

S3 records collected: 3


,scenario,N_intervals,seed,chosen,n_clusters,noise_pct,chi_min,chi_max
0,S3,3,42,False,7,7.5,0.134,0.861
1,S3,3,123,True,7,7.5,0.134,0.861
2,S3,3,456,False,7,7.5,0.134,0.861


In [26]:
# ── S3: Loop over all (N, seed) configurations ───────────────────────────────

s3_pkl_dir = os.path.join(s3_root, 'data', 'clustering_results/')

N_intervals_s3 = [3, 4]      # 3 = alternative, 4 = paper choice
seeds          = [42, 123, 456]
paper_config   = (3, 123)    # (N, seed) used in the paper

records_s3 = []

for N in N_intervals_s3:
    for seed in seeds:
        pkl_path = os.path.join(s3_pkl_dir, f'Nint{N}_seed{seed}_FNs.pkl')
        
        if not os.path.exists(pkl_path):
            print(f'  MISSING: {pkl_path}')
            continue
        
        FNs    = load_fns(pkl_path)
        stats  = cluster_stats(FNs, chi0_s3)
        chosen = (N, seed) == paper_config
        
        records_s3.append({
            'scenario':    'S3',
            'N_intervals': N,
            'seed':        seed,
            'chosen':      chosen,
            **stats
        })
        
        marker = '  [PAPER]' if chosen else ''
        print(f'  N={N} seed={seed}{marker} | '
              f'clusters={stats["n_clusters"]} '
              f'noise={stats["noise_pct"]:.1f}% '
              f'chi=[{stats["chi_min"]:.3f}, {stats["chi_max"]:.3f}]')

df_s3 = pd.DataFrame(records_s3)
print(f'\nS3 records collected: {len(df_s3)}')
df_s3.drop(columns=['chi_per_cluster']).round(3)

  N=3 seed=42 | clusters=7 noise=7.5% chi=[0.134, 0.861]
  N=3 seed=123  [PAPER] | clusters=7 noise=7.5% chi=[0.134, 0.861]
  N=3 seed=456 | clusters=7 noise=7.5% chi=[0.134, 0.861]
  N=4 seed=42 | clusters=9 noise=15.7% chi=[0.122, 0.885]
  N=4 seed=123 | clusters=9 noise=15.4% chi=[0.122, 0.888]
  N=4 seed=456 | clusters=8 noise=15.9% chi=[0.121, 0.885]

S3 records collected: 6


,scenario,N_intervals,seed,chosen,n_clusters,noise_pct,chi_min,chi_max
0,S3,3,42,False,7,7.5,0.134,0.861
1,S3,3,123,True,7,7.5,0.134,0.861
2,S3,3,456,False,7,7.5,0.134,0.861
3,S3,4,42,False,9,15.7,0.122,0.885
4,S3,4,123,False,9,15.4,0.122,0.888
5,S3,4,456,False,8,15.9,0.121,0.885


---
## 5. Compile full DataFrame

In [27]:
df_all = pd.concat([df_s1, df_s2, df_s3], ignore_index=True)

# Flag problematic runs
NOISE_FLAG    = 15.0   # %
BROWN_MAX_CHI = 0.20   # brown macrostate: chi_min must be below this
GREEN_MIN_CHI = 0.60   # green macrostate: chi_max must be above this

df_all['flagged'] = (
    (df_all['noise_pct']  > NOISE_FLAG)    |
    (df_all['chi_min']   >= BROWN_MAX_CHI) |
    (df_all['chi_max']   <= GREEN_MIN_CHI)
)

# Save
df_all.drop(columns=['chi_per_cluster'], errors='ignore').to_csv(
    'stability_table_all_scenarios.csv', index=False
)

print(f'Total runs collected: {len(df_all)}')
print(f'Flagged runs: {df_all.flagged.sum()}')
df_all.drop(columns=['chi_per_cluster', 'flagged'], errors='ignore').round(3)

Total runs collected: 18
Flagged runs: 7


,scenario,N_intervals,seed,chosen,n_clusters,noise_pct,chi_min,chi_max
0,S1,3,42,False,6,6.6,0.161,0.694
1,S1,3,123,False,4,35.3,0.161,0.692
2,S1,3,456,False,6,6.6,0.161,0.694
3,S1,4,42,False,11,7.4,0.137,0.713
4,S1,4,123,True,10,7.4,0.136,0.713
5,S1,4,456,False,10,7.4,0.136,0.713
6,S2,3,42,False,13,7.5,0.202,0.697
7,S2,3,123,True,13,7.5,0.201,0.692
8,S2,3,456,False,13,7.5,0.201,0.692
9,S2,4,42,False,25,7.0,0.181,0.746


In [28]:
print(df_all.drop(columns=['chi_per_cluster', 'flagged'], errors='ignore').round(3))

   scenario  N_intervals  seed  chosen  n_clusters  noise_pct  chi_min  \
0        S1            3    42   False           6        6.6    0.161   
1        S1            3   123   False           4       35.3    0.161   
2        S1            3   456   False           6        6.6    0.161   
3        S1            4    42   False          11        7.4    0.137   
4        S1            4   123    True          10        7.4    0.136   
5        S1            4   456   False          10        7.4    0.136   
6        S2            3    42   False          13        7.5    0.202   
7        S2            3   123    True          13        7.5    0.201   
8        S2            3   456   False          13        7.5    0.201   
9        S2            4    42   False          25        7.0    0.181   
10       S2            4   123   False          17        7.1    0.184   
11       S2            4   456   False          26        7.0    0.181   
12       S3            3    42   False

---
## 6. Summary table
Summarises each (scenario, N_intervals) group across seeds.

In [29]:
print('\n' + '='*70)
print('ROBUSTNESS SUMMARY')
print('='*70)

scenario_meta = [
    ('S1', 'Voter dynamics (fully connected)', 4, [3, 4]),
    ('S2', 'Voter dynamics (small-world)',      3, [3,4]),
    ('S3', 'Majority rule',                     3, [3,4]),
]

for scen_id, scen_label, N_chosen, N_list in scenario_meta:
    sub = df_all[df_all.scenario == scen_id]
    if sub.empty:
        print(f'\n{scen_label}: no data.')
        continue
    print(f'\n{scen_label}  (chosen N={N_chosen})')
    for N in N_list:
        rows = sub[sub.N_intervals == N].dropna(subset=['n_clusters'])
        if rows.empty:
            print(f'  N={N}: — no data —'); continue
        n_ok   = (~rows.flagged).sum()
        n_flag = rows.flagged.sum()
        marker = ' ← paper' if N == N_chosen else ''
        print(
            f'  N={N}{marker}: {n_ok}/{len(rows)} runs OK, {n_flag} flagged | '
            f'clusters {int(rows.n_clusters.min())}–{int(rows.n_clusters.max())} | '
            f'noise {rows.noise_pct.min():.1f}–{rows.noise_pct.max():.1f}% | '
            f'χ-min [{rows.chi_min.min():.3f}–{rows.chi_min.max():.3f}] | '
            f'χ-max [{rows.chi_max.min():.3f}–{rows.chi_max.max():.3f}]'
        )
        for _, r in rows[rows.flagged].iterrows():
            reasons = []
            if r.noise_pct > NOISE_FLAG:   reasons.append(f'noise={r.noise_pct:.1f}%')
            if r.chi_min >= BROWN_MAX_CHI: reasons.append(f'chi_min={r.chi_min:.3f}')
            if r.chi_max <= GREEN_MIN_CHI: reasons.append(f'chi_max={r.chi_max:.3f}')
            print(f'    → seed={r.seed} flagged: {"; ".join(reasons)}')


ROBUSTNESS SUMMARY

Voter dynamics (fully connected)  (chosen N=4)
  N=3: 2/3 runs OK, 1 flagged | clusters 4–6 | noise 6.6–35.3% | χ-min [0.161–0.161] | χ-max [0.692–0.694]
    → seed=123 flagged: noise=35.3%
  N=4 ← paper: 3/3 runs OK, 0 flagged | clusters 10–11 | noise 7.4–7.4% | χ-min [0.136–0.137] | χ-max [0.713–0.713]

Voter dynamics (small-world)  (chosen N=3)
  N=3 ← paper: 0/3 runs OK, 3 flagged | clusters 13–13 | noise 7.5–7.5% | χ-min [0.201–0.202] | χ-max [0.692–0.697]
    → seed=42 flagged: chi_min=0.202
    → seed=123 flagged: chi_min=0.201
    → seed=456 flagged: chi_min=0.201
  N=4: 3/3 runs OK, 0 flagged | clusters 17–26 | noise 7.0–7.1% | χ-min [0.181–0.184] | χ-max [0.746–0.752]

Majority rule  (chosen N=3)
  N=3 ← paper: 3/3 runs OK, 0 flagged | clusters 7–7 | noise 7.5–7.5% | χ-min [0.134–0.134] | χ-max [0.861–0.861]
  N=4: 0/3 runs OK, 3 flagged | clusters 8–9 | noise 15.4–15.9% | χ-min [0.121–0.122] | χ-max [0.885–0.888]
    → seed=42 flagged: noise=15.7%
    → 

---
## 7. LaTeX table for the paper

In [30]:
def fmt_cell(row):
    """Format one table cell: clusters / noise% / [chi_min, chi_max]"""
    if pd.isna(row.get('n_clusters')):
        return '—'
    val = (f"{int(row['n_clusters'])} / {row['noise_pct']:.1f}\\% / "
           f"[{row['chi_min']:.3f}, {row['chi_max']:.3f}]")
    if row.get('chosen'):
        val = r'\textbf{' + val + r'}'
    if row.get('flagged'):
        val += r' $\dagger$'
    return val


table_config = [
    ('S1', r'Voter dynamics\\(fully connected)', 4, [3, 4]),
    ('S2', r'Voter dynamics\\(small-world)',      3, [3]),
    ('S3', r'Majority rule',                      3, [3]),
]

lines = [
    r'\begin{table}[ht]',
    r'\centering\small\setlength{\tabcolsep}{4pt}',
    (r'\caption{Robustness of clustering across interval choices ($N_{\text{int}}$) '
     r'and random seeds. Each cell: total clusters / noise\% / '
     r'[$\hat{\chi}_{\min}$, $\hat{\chi}_{\max}$]. '
     r'\textbf{Bold} = configuration used in the paper. '
     r'$\dagger$ = flagged run (noise $>20\%$ or macrostate endpoint not recovered).}'),
    r'\label{tab:robustness}',
    r'\begin{tabular}{llccc}',
    r'\toprule',
    r'Scenario & $N_{\text{int}}$ & Seed 42 & Seed 123 & Seed 456 \\',
    r'\midrule',
]

for s_idx, (scen_id, label, N_chosen, N_list) in enumerate(table_config):
    sub = df_all[df_all.scenario == scen_id]
    for i, N in enumerate(N_list):
        cells = []
        for seed in [42, 123, 456]:
            r = sub[(sub.N_intervals == N) & (sub.seed == seed)]
            cells.append(fmt_cell(r.iloc[0].to_dict()) if not r.empty else '—')
        scen_col = (
            r'\multirow{' + str(len(N_list)) + r'}{*}{\shortstack[l]{' + label + r'}}'
        ) if i == 0 else ''
        lines.append(f'{scen_col} & {N} & {cells[0]} & {cells[1]} & {cells[2]} \\\\')
    lines.append(r'\midrule' if s_idx < len(table_config)-1 else r'\bottomrule')

lines += [r'\end{tabular}', r'\end{table}']
latex_str = '\n'.join(lines)

with open('stability_table_paper.txt', 'w') as f:
    f.write(latex_str)

print('LaTeX table saved → stability_table_paper.txt')
print('\n--- Preview ---')
print(latex_str)

LaTeX table saved → stability_table_paper.txt

--- Preview ---
\begin{table}[ht]
\centering\small\setlength{\tabcolsep}{4pt}
\caption{Robustness of clustering across interval choices ($N_{\text{int}}$) and random seeds. Each cell: total clusters / noise\% / [$\hat{\chi}_{\min}$, $\hat{\chi}_{\max}$]. \textbf{Bold} = configuration used in the paper. $\dagger$ = flagged run (noise $>20\%$ or macrostate endpoint not recovered).}
\label{tab:robustness}
\begin{tabular}{llccc}
\toprule
Scenario & $N_{\text{int}}$ & Seed 42 & Seed 123 & Seed 456 \\
\midrule
\multirow{2}{*}{\shortstack[l]{Voter dynamics\\(fully connected)}} & 3 & 6 / 6.6\% / [0.161, 0.694] & 4 / 35.3\% / [0.161, 0.692] $\dagger$ & 6 / 6.6\% / [0.161, 0.694] \\
 & 4 & 11 / 7.4\% / [0.137, 0.713] & \textbf{10 / 7.4\% / [0.136, 0.713]} & 10 / 7.4\% / [0.136, 0.713] \\
\midrule
\multirow{1}{*}{\shortstack[l]{Voter dynamics\\(small-world)}} & 3 & 13 / 7.5\% / [0.202, 0.697] $\dagger$ & \textbf{13 / 7.5\% / [0.201, 0.692]} $\dagger$